# step6 보강 — 이름이 멀쩡한가 · 방향이 층 특이적인가

**어느 스텝:** step6 보강. 본실험(`step6_steer.ipynb`)을 돌린 뒤에 한다.

## 왜 필요한가

본실험에서 **회복률이 1을 크게 넘었다**(DeepSeek 세기4에서 4.29).
회복률 1이 "문맥이 깨끗한 상태(천장)"인데 그 4배다. 이건 되살아난 게 아니라 **과잉 조향**이다.

선호 점수는 위가 막혀 있지 않아 세게 밀수록 계속 오른다. 그래서 **점수만으로는
"되살아났다"고 말할 수 없다.** 실제로 나온 이름이 멀쩡한지 봐야 한다.

또 **엉뚱한 층(L5)도 맞는 층의 64%만큼 작동했다.** 지금 대조는 "그 층에서 뽑은 방향을
그 층에 넣기"라, 초반 층에도 camel 방향이 있으면 당연히 된다. 더 날카로운 질문은
**"맞는 층에서 뽑은 방향을 엉뚱한 층에 넣어도 되나"** 다.

## 두 가지를 잰다

| | 무엇 | 왜 |
|---|---|---|
| **가. 생성 검증** | 조향을 건 채로 실제 이름을 생성 → 표기·건전성 | 점수만 오르고 이름이 깨졌는지 확인 |
| **나. 층 특이성** | 맞는 층 방향을 엉뚱한 층에 주입 | 방향 자체가 층 특이적인가 |

**이름 건전성 판정:** 식별자 형식 · 길이 2~40 · 같은 조각 반복 없음
(`removeDuplicatesDuplicates` 같은 것을 잡는다).

## 결과가 어느 쪽으로 나오든 무슨 뜻인지

| 나오는 그림 | 뜻 |
|---|---|
| 세기 올려도 이름 멀쩡, 준수율 오름 | **처방이 진짜 작동한다.** 회복률 곡선을 그대로 쓴다 |
| 세기 4~8에서 이름이 깨짐 | **최적 세기가 있다.** 깨지기 직전 값을 보고한다 |
| 교차 주입도 잘 됨 | 방향은 층을 안 가린다 → **"어느 층이든 된다"로 서술** |
| 교차 주입이 안 됨 | **방향이 층 특이적** → 층 주장이 살아난다 |

**부하:** 생성이라 본실험보다 조건당 느리다. 묶음을 21개로 줄여 놓았다.

In [ ]:
# ② 환경
!pip install -q -r requirements.txt
import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (매우 느림)')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# ③ 저장소
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
BRANCH = 'integration/step1-5'
!git fetch --quiet origin $BRANCH
!git checkout $BRANCH
!git pull --quiet origin $BRANCH
!pip install -e . -q
import sys; sys.path.insert(0, 'src')
print('브랜치:', BRANCH)

In [ ]:
# ④ 조건 설정
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation,
                                Intervention, InterventionKind)

MODELS = [
    ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct',           family='qwen',      dtype='float16'),
    ModelSpec(name='deepseek-ai/deepseek-coder-6.7b-instruct',  family='deepseek',  dtype='float16'),
    ModelSpec(name='unsloth/Llama-3.2-3B-Instruct',             family='llama',     dtype='float16'),
    ModelSpec(name='stabilityai/stable-code-instruct-3b',       family='stability', dtype='float16'),
]
PEAK  = {'qwen': 25, 'deepseek': 20, 'llama': 15, 'stability': 18}
EARLY = {'qwen': 5,  'deepseek': 5,  'llama': 3,  'stability': 4}

# ★ 이번에 돌릴 모델 (0=qwen, 1=deepseek, 2=llama, 3=stable)
PICK = 0
MODEL = MODELS[PICK]
peak, early = PEAK[MODEL.family], EARLY[MODEL.family]
print('이번 모델:', MODEL.family, '| 맞는 층 L', peak, '| 엉뚱한 층 L', early)

BLOCKS    = list(range(21))            # 본실험 42의 절반 — 생성이라 느리다
STRENGTHS = [1.0, 2.0, 4.0, 8.0]
INSTR = Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL)

def base(block):
    return dict(model=MODEL,
                preceding=PrecedingCode(n_compliant=0, n_functions=12,
                                        composition=Composition.POOL, pool_block=block),
                instruction=INSTR, seed=SEED, tag='cliff')

# 가. 생성 검증 — 무개입 + 맞는 층 세기 스윕
gen_conditions = []
for b in BLOCKS:
    gen_conditions.append(Condition(**base(b), intervention=Intervention()))
    for s in STRENGTHS:
        gen_conditions.append(Condition(**base(b), intervention=Intervention(
            kind=InterventionKind.VALUE_ADD, layers=[peak],
            strength=s, steer_source='code_contrast')))

# 나. 층 특이성 — 맞는 층에서 뽑은 방향을 엉뚱한 층에 주입
cross_conditions = []
for b in BLOCKS:
    for s in STRENGTHS:
        cross_conditions.append(Condition(**base(b), intervention=Intervention(
            kind=InterventionKind.VALUE_ADD, layers=[early],
            strength=s, steer_source='code_contrast', steer_layer=peak)))

print(f'생성 검증 {len(gen_conditions)}개 · 층 특이성 {len(cross_conditions)}개')
assert len({c.slug() for c in gen_conditions}) == len(gen_conditions)
assert len({c.slug() for c in cross_conditions}) == len(cross_conditions)

In [ ]:
# ⑤ 실행
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model
import numpy as np
from collections import defaultdict

jobs = [('step6_steer-generate', gen_conditions, 'steer_generate'),
        ('step6_steer-crosslayer', cross_conditions, 'steer')]
todo_all = [(step, c, mode) for step, cs, mode in jobs for c in cs
            if not result_path(c, step=step).exists()]
print(f'[{MODEL.family}] 남은 조건 {len(todo_all)}개')

if todo_all:
    handle = load_model(MODEL)
    seen = defaultdict(list)
    for i, (step, c, mode) in enumerate(todo_all, 1):
        out = run(c, handle=handle, mode=mode, max_new_tokens=24)
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=step, rq='RQ2/RQ3'))
        ex = out.metrics.extra
        s = ex.get('strength')
        if mode == 'steer_generate':
            key = f"생성 세기{s:g}" if s else '생성 무개입'
            seen[key].append((ex['compliant'], ex['name_ok']))
        elif not ex['undecidable'] and ex['recovery'] is not None:
            seen[f'교차 세기{s:g}'].append((ex['recovery'], True))
        del out
        if i % 30 == 0 or i == len(todo_all):
            parts = []
            for k in sorted(seen):
                v = seen[k]
                if k.startswith('생성'):
                    parts.append(f"{k} 준수{np.mean([a for a,_ in v]):.2f}/멀쩡{np.mean([b for _,b in v]):.2f}")
                else:
                    parts.append(f"{k} {np.mean([a for a,_ in v]):.2f}")
            print(f'  [{i}/{len(todo_all)}] ' + ' | '.join(parts))
    del handle
    import torch, gc; gc.collect(); torch.cuda.empty_cache()
print('완료')

In [ ]:
# ⑥ 요약 가 — 세게 밀면 이름이 깨지는가
import numpy as np
from collections import defaultdict
from harness.results import load_result
from harness import result_path

recs = [load_result(result_path(c, step='step6_steer-generate')) for c in gen_conditions
        if result_path(c, step='step6_steer-generate').exists()]
by = defaultdict(list)
for r in recs:
    ex = r.metrics.extra
    by[ex['strength'] if ex['strength'] else 0.0].append(ex)

print(f"[{MODEL.family}] 맞는 층 L{peak} — 조향 세기별 실제 생성 결과\n")
print(f"{'세기':>6}{'준수율':>9}{'멀쩡한 이름':>12}{'camel':>8}{'snake':>8}{'그 외':>8}{'n':>5}")
for s in sorted(by):
    v = by[s]
    nota = [e['notation'] for e in v]
    print(f"{s:>6g}{np.mean([e['compliant'] for e in v]):>9.3f}"
          f"{np.mean([e['name_ok'] for e in v]):>12.3f}"
          f"{nota.count('camel')/len(v):>8.2f}{nota.count('snake')/len(v):>8.2f}"
          f"{nota.count('other')/len(v):>8.2f}{len(v):>5}")

print('\n깨진 이름 예시:')
bad = [r.metrics.extra for r in recs if not r.metrics.extra['name_ok']][:6]
if not bad:
    print('  (없음 — 모든 이름이 멀쩡하다)')
for e in bad:
    print(f"  세기 {e['strength'] or 0:g}: {e['name']!r} — {e['name_reason']}")
print('\n읽는 법: 세기를 올려도 「멀쩡한 이름」이 1에 가깝고 준수율이 오르면 처방이 진짜다.')
print('        어느 세기부터 멀쩡한 이름이 떨어지면 거기가 한계다.')

In [ ]:
# ⑦ 요약 나 — 방향이 층 특이적인가
import numpy as np
from collections import defaultdict
from harness.results import load_result
from harness import result_path

cross = [load_result(result_path(c, step='step6_steer-crosslayer')) for c in cross_conditions
         if result_path(c, step='step6_steer-crosslayer').exists()]
cr = defaultdict(list)
n_und = 0
for r in cross:
    ex = r.metrics.extra
    if ex['undecidable'] or ex['recovery'] is None:
        n_und += 1; continue
    cr[ex['strength']].append(ex['recovery'])

# 본실험에서 같은 층 주입 값을 불러와 나란히
import glob, json as _json
same_peak, same_early = defaultdict(list), defaultdict(list)
for f in glob.glob('results/step6_steer/*.json'):
    d = _json.load(open(f))
    if d['condition']['model']['family'] != MODEL.family: continue
    ex = d['metrics']['extra']
    if ex['method'] != 'value_add' or ex['undecidable'] or ex['recovery'] is None: continue
    (same_peak if ex['layer'] == peak else same_early)[ex['strength']].append(ex['recovery'])

print(f"[{MODEL.family}] 판정 불가 {n_und}개\n")
print(f"{'세기':>6}{'맞는층→맞는층':>15}{'엉뚱층→엉뚱층':>15}{'맞는층→엉뚱층':>15}")
for s in sorted(set(cr) | set(same_peak)):
    f = lambda d: f'{np.mean(d[s]):.3f}' if d.get(s) else '—'
    print(f"{s:>6g}{f(same_peak):>15}{f(same_early):>15}{f(cr):>15}")
print('\n읽는 법: 맨 오른쪽(교차)이 낮으면 **방향이 층 특이적**이다.')
print('        맨 오른쪽도 높으면 방향은 층을 안 가린다 — 층 주장을 좁혀야 한다.')

In [ ]:
STEPS = ['step6_steer-generate', 'step6_steer-crosslayer']
# ⑧ 결과 zip으로 묶어 내려받기
import shutil, os, glob

def pack(step):
    d = f'results/{step}'
    if not os.path.isdir(d):
        print(f'  [건너뜀] {d} 폴더가 없다 — 이 스텝은 아직 안 돌렸다')
        return None
    n = len(glob.glob(f'{d}/*.json'))
    if n == 0:
        print(f'  [건너뜀] {d} 가 비어 있다')
        return None
    path = shutil.make_archive(step, 'zip', d)
    print(f'  {step}: {n}개 → {path} ({os.path.getsize(path)/1e6:.1f}MB)')
    return path

print('results/ 안에 있는 폴더:', sorted(os.listdir('results')) if os.path.isdir('results') else '(results 폴더 없음)')
print()
made = [p for p in (pack(s) for s in STEPS) if p]

if not made:
    print('\n내려받을 것이 없다. 실행 셀(⑤)을 먼저 돌렸는지 확인할 것.')
else:
    try:
        from google.colab import files
        for p in made:
            files.download(p)
        print('\n다운로드 시작. 브라우저가 막으면 왼쪽 **파일 탐색기**에서 직접 받으면 된다.')
    except Exception as e:
        print(f'\nColab 자동 다운로드 불가({type(e).__name__}). 위 경로에서 직접 받을 것.')
